# Modeling lab
Experiments on the patient risk model, kept apart from the main analysis in
`sparcs_sepsis_2023.ipynb`. Same rules as there: out-of-fold predictions split
by hospital, and calibration comes first.

1. Model comparison on the main feature set
2. Feature experiments: interactions
3. Is any model really better? Bootstrap by hospital
4. Explainability

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # data/ and figs/ paths are relative to the repo root

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score

from sparcs import models
from sparcs.config import SEED
from sparcs.data import build_cohort, clean, load_raw

df, _ = build_cohort(clean(load_raw()))
y = df["died"].to_numpy()
groups = df["permanent_facility_id"].to_numpy()
cols = models.FEATURE_SETS["full"]
X = models.features(df, cols)

## 1. Model comparison (main feature set)
- **LR, L2** (sklearn's default mild penalty, C=1): the main pipeline's model
- **LR, unpenalized** (C=inf): the classic risk-adjustment model
- **LR, L1** at two strengths: smaller C = stronger penalty, more categories pushed to exactly 0
- **LightGBM**: flexible trees, finds interactions on its own
- **EBM** (Explainable Boosting Machine): boosting, but one shape per feature plus a few
  pairwise interactions, so every effect can be read off directly

In [2]:
def ebm():
    return ExplainableBoostingClassifier(random_state=SEED, n_jobs=-1)


candidates = {
    "lr_l2": models.logreg(cols),
    "lr_unpenalized": models.logreg(cols, C=np.inf),
    "lr_l1_C0.01": models.logreg(cols, l1_ratio=1, C=0.01, solver="saga"),
    "lr_l1_C0.001": models.logreg(cols, l1_ratio=1, C=0.001, solver="saga"),
    "ebm": ebm,
}
preds = {name: models.oof_predict(make, X, y, groups) for name, make in candidates.items()}
# LightGBM reads pandas categories natively
preds["lgbm"] = models.oof_predict(models.lgbm, X.astype("category"), y, groups)

models.metrics_table(y, preds)

,model,auroc,auprc,brier,cal_intercept,cal_slope
0,lr_l2,0.8081,0.2945,0.0937,-0.0020,0.9906
1,lr_unpenalized,0.8081,0.2945,0.0937,-0.0012,0.9899
2,lr_l1_C0.01,0.8061,0.2897,0.0938,-0.0017,1.0585
3,lr_l1_C0.001,0.7927,0.2678,0.0950,-0.0018,1.3995
4,ebm,0.8089,0.2948,0.0936,-0.0016,0.9840
5,lgbm,0.8076,0.2921,0.0936,-0.0013,0.9709


## 2. Feature experiments: interactions
LightGBM and EBM can learn interactions on their own. Does logistic regression
catch up if it's handed a few likely ones?
- age x severity: severe illness may be deadlier in older patients
- ROM x admission type: an elective admission with Extreme ROM is unusual

In [3]:
df_int = df.assign(
    age_x_severity=df["age_group"] + " | " + df["apr_severity_of_illness_code"].astype(str),
    rom_x_admission=df["apr_risk_of_mortality"] + " | " + df["type_of_admission"],
)
int_cols = cols + ["age_x_severity", "rom_x_admission"]
preds["lr_l2_interactions"] = models.oof_predict(
    models.logreg(int_cols), models.features(df_int, int_cols), y, groups)

models.metrics_table(y, {k: preds[k] for k in ["lr_l2", "lr_l2_interactions", "lgbm", "ebm"]})

,model,auroc,auprc,brier,cal_intercept,cal_slope
0,lr_l2,0.8081,0.2945,0.0937,-0.0020,0.9906
1,lr_l2_interactions,0.8087,0.2961,0.0936,-0.0016,0.9884
2,lgbm,0.8076,0.2921,0.0936,-0.0013,0.9709
3,ebm,0.8089,0.2948,0.0936,-0.0016,0.9840


## 3. Is any model really better? Bootstrap by hospital
Differences in the 4th decimal could be noise. Resample **hospitals** (not
patients) with replacement, since patients in the same hospital aren't
independent, and get a 95% CI for each model's difference from `lr_l2`.
An interval that contains 0 means no real difference.
- d_auroc > 0: better at ranking patients
- d_brier < 0: more accurate probabilities

In [4]:
def bootstrap_diff(y, p_base, p_other, groups, n_boot=200, seed=SEED):
    rng = np.random.default_rng(seed)
    hospitals = np.unique(groups)
    rows_of = {h: np.flatnonzero(groups == h) for h in hospitals}
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([rows_of[h] for h in rng.choice(hospitals, len(hospitals))])
        diffs.append([
            roc_auc_score(y[idx], p_other[idx]) - roc_auc_score(y[idx], p_base[idx]),
            brier_score_loss(y[idx], p_other[idx]) - brier_score_loss(y[idx], p_base[idx]),
        ])
    lo, hi = np.percentile(diffs, [2.5, 97.5], axis=0)
    return {"d_auroc_low": lo[0], "d_auroc_high": hi[0], "d_brier_low": lo[1], "d_brier_high": hi[1]}


pd.DataFrame({name: bootstrap_diff(y, preds["lr_l2"], p, groups)
              for name, p in preds.items() if name != "lr_l2"}).T.round(4)

,d_auroc_low,d_auroc_high,d_brier_low,d_brier_high
lr_unpenalized,-0.0001,0.0001,-0.0000,0.0000
lr_l1_C0.01,-0.0030,-0.0009,-0.0000,0.0002
lr_l1_C0.001,-0.0185,-0.0111,0.0008,0.0018
ebm,0.0001,0.0016,-0.0002,-0.0001
lgbm,-0.0018,0.0007,-0.0002,0.0001
lr_l2_interactions,0.0001,0.0012,-0.0002,-0.0001


## 4. Explainability
Odds ratios for the main model are in the main notebook (section 3b). Here: what
the other models rely on.

In [5]:
# EBM fitted on all patients: average size of each term's effect (log-odds scale).
# Terms with " & " are pairwise interactions EBM chose to add.
ebm_all = ebm().fit(X, y)
pd.Series(ebm_all.term_importances(), index=ebm_all.term_names_).sort_values(ascending=False).round(3)

apr_risk_of_mortality                                            1.228
apr_severity_of_illness_code                                     0.589
age_group                                                        0.252
age_group & apr_severity_of_illness_code                         0.236
age_group & apr_risk_of_mortality                                0.203
apr_severity_of_illness_code & apr_risk_of_mortality             0.061
emergency_department_indicator                                   0.044
type_of_admission & apr_severity_of_illness_code                 0.035
gender & apr_severity_of_illness_code                            0.034
emergency_department_indicator & apr_severity_of_illness_code    0.026
type_of_admission                                                0.020
gender                                                           0.018
emergency_department_indicator & apr_risk_of_mortality           0.018
gender & apr_risk_of_mortality                                   0.009
type_o

In [6]:
# EBM's learned effect for each level of risk of mortality (log-odds added to the prediction)
i = ebm_all.term_names_.index("apr_risk_of_mortality")
levels = ebm_all.bins_[i][0]  # category -> bin index
pd.Series({level: ebm_all.term_scores_[i][b] for level, b in levels.items()}).sort_values().round(2)

Minor      -3.71
Moderate   -1.53
Major      -0.17
Extreme     1.36
dtype: float64

In [7]:
# LightGBM: total gain from each feature's splits (how much each one improved the fit)
lgb_all = models.lgbm().fit(X.astype("category"), y)
pd.Series(lgb_all.booster_.feature_importance("gain"), index=cols).sort_values(ascending=False).round(0)

apr_risk_of_mortality             289090.0
apr_severity_of_illness_code       25843.0
age_group                          11473.0
emergency_department_indicator      4822.0
type_of_admission                   2109.0
gender                              1465.0
dtype: float64

In [8]:
# Strong L1 penalty: which categories survive, and which are pushed to exactly 0?
l1 = models.logreg(cols, l1_ratio=1, C=0.001, solver="saga")().fit(X, y)
coef = pd.Series(l1.named_steps["lr"].coef_[0], index=l1.named_steps["pre"].get_feature_names_out())
print(f"{(coef == 0).sum()} of {len(coef)} categories set to exactly 0")
coef[coef != 0].sort_values().round(3)

20 of 23 categories set to exactly 0


cat__age_group_70 or Older             0.134
cat__apr_severity_of_illness_code_4    0.907
cat__apr_risk_of_mortality_Extreme     1.409
dtype: float64